# Learner Classification (Slow / Medium / Fast) — ANN + Model Comparison

This notebook trains an **Artificial Neural Network (ANN)** to classify students as
**Slow / Medium / Fast learners** from their quiz behaviour, and benchmarks it against
four classical algorithms (Random Forest, Gradient Boosting, SVM, Logistic Regression).

**Features used:** `msq` (marks), `TTQ_sec` (time in seconds), `accuracy`, `HIT_encoded` (mistake severity)
**Target:** `label_final_KMeans`

> Note: the labels were derived from these quiz attributes via K-Means, so the task is to
> *learn the labelling rule*. Very high accuracy across all models is therefore expected and
> desirable — it confirms the classifier faithfully reproduces the Slow/Medium/Fast scheme
> for new students.

In [ ]:
import os, json, pickle, warnings
warnings.filterwarnings('ignore'); os.environ['TF_CPP_MIN_LOG_LEVEL']='3'
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.svm import SVC
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (accuracy_score, f1_score, precision_score, recall_score,
                             classification_report, confusion_matrix, ConfusionMatrixDisplay)
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout, Input
from tensorflow.keras.callbacks import EarlyStopping
SEED=42; np.random.seed(SEED); tf.random.set_seed(SEED)

## 1. Load data & define features

In [ ]:
DATA_PATH='DS1_FINAL_labeled_for_ANN.xlsx'
FEATURES=['msq','TTQ_sec','accuracy','HIT_encoded']; TARGET='label_final_KMeans'
df=pd.read_excel(DATA_PATH, sheet_name='Sheet1')
X=df[FEATURES].values.astype('float32')
le=LabelEncoder(); y=le.fit_transform(df[TARGET])
print('Classes:', list(le.classes_)); print('Class counts:\n', df[TARGET].value_counts())
df[FEATURES+[TARGET]].head()

## 2. Train / Validation / Test split (70/15/15, stratified) + scaling

In [ ]:
X_tr,X_tmp,y_tr,y_tmp=train_test_split(X,y,test_size=0.30,stratify=y,random_state=SEED)
X_val,X_te,y_val,y_te=train_test_split(X_tmp,y_tmp,test_size=0.50,stratify=y_tmp,random_state=SEED)
scaler=StandardScaler().fit(X_tr)
Xtr,Xval,Xte,Xall=(scaler.transform(a) for a in (X_tr,X_val,X_te,X))
print('train/val/test =', len(y_tr), len(y_val), len(y_te))

## 3. Classical baselines (5-fold CV + held-out test)

In [ ]:
def mm(yt,yp): return dict(accuracy=accuracy_score(yt,yp),
    f1_macro=f1_score(yt,yp,average='macro'),
    precision_macro=precision_score(yt,yp,average='macro'),
    recall_macro=recall_score(yt,yp,average='macro'))
skf=StratifiedKFold(5,shuffle=True,random_state=SEED); results={}
baselines={'Random Forest':RandomForestClassifier(n_estimators=300,random_state=SEED),
 'Gradient Boosting':GradientBoostingClassifier(random_state=SEED),
 'SVM (RBF)':SVC(kernel='rbf',C=10,gamma='scale',probability=True,random_state=SEED),
 'Logistic Regression':LogisticRegression(max_iter=1000,random_state=SEED)}
fitted={}
for n,clf in baselines.items():
    clf.fit(Xtr,y_tr); fitted[n]=clf
    cv=cross_val_score(clf,Xall,y,cv=skf)
    results[n]={'test':mm(y_te,clf.predict(Xte)),'cv_mean':cv.mean(),'cv_std':cv.std()}
    print(f'{n:22s} test_acc={results[n]["test"]["accuracy"]*100:6.2f}  cv={cv.mean()*100:5.2f} ± {cv.std()*100:.2f}')

## 4. Build & train the ANN (MLP)

In [ ]:
def build_ann(indim,ncls):
    m=Sequential([Input((indim,)),Dense(32,activation='relu'),Dropout(0.2),
                  Dense(16,activation='relu'),Dropout(0.1),Dense(ncls,activation='softmax')])
    m.compile('adam','sparse_categorical_crossentropy',metrics=['accuracy']); return m
ann=build_ann(Xtr.shape[1],len(le.classes_))
es=EarlyStopping(monitor='val_loss',patience=15,restore_best_weights=True)
hist=ann.fit(Xtr,y_tr,validation_data=(Xval,y_val),epochs=200,batch_size=32,verbose=0,callbacks=[es])
y_pred=ann.predict(Xte,verbose=0).argmax(1)
results['ANN (MLP)']={'test':mm(y_te,y_pred),'epochs':len(hist.history['loss'])}
print('ANN test accuracy: %.2f%%  (epochs=%d)'%(results['ANN (MLP)']['test']['accuracy']*100,len(hist.history['loss'])))

## 5. Learning curves

In [ ]:
fig,ax=plt.subplots(1,2,figsize=(10,3.5))
ax[0].plot(hist.history['accuracy'],label='train'); ax[0].plot(hist.history['val_accuracy'],label='val')
ax[0].set_title('Accuracy'); ax[0].set_xlabel('epoch'); ax[0].legend()
ax[1].plot(hist.history['loss'],label='train'); ax[1].plot(hist.history['val_loss'],label='val')
ax[1].set_title('Loss'); ax[1].set_xlabel('epoch'); ax[1].legend(); plt.tight_layout(); plt.show()

## 6. ANN per-class report & confusion matrix

In [ ]:
print(classification_report(y_te,y_pred,target_names=le.classes_))
ConfusionMatrixDisplay(confusion_matrix(y_te,y_pred),display_labels=le.classes_).plot(cmap='Blues')
plt.title('ANN — Confusion Matrix (test)'); plt.xticks(rotation=30,ha='right'); plt.show()

## 7. Final comparison table

In [ ]:
rows=[]
for n,r in results.items():
    rows.append({'Model':n,'Test Acc %':round(r['test']['accuracy']*100,2),
        'Test F1 %':round(r['test']['f1_macro']*100,2),
        'CV %':round(r.get('cv_mean',float('nan'))*100,2) if 'cv_mean' in r else None})
pd.DataFrame(rows).sort_values('Test Acc %',ascending=False).reset_index(drop=True)

## 8. Save the chosen model (ANN) for deployment

The ANN is exported in **.keras** and **.h5**; the scaler + label encoder needed to run it
are saved in **preprocessing.pkl**. The best classical baseline (Random Forest) is also
pickled for reference.

In [ ]:
ann.save('learner_ann_model.keras'); ann.save('learner_ann_model.h5')
pickle.dump({'scaler':scaler,'label_encoder':le,'feature_names':FEATURES},open('preprocessing.pkl','wb'))
pickle.dump(fitted['Random Forest'],open('best_rf_model.pkl','wb'))
print('Saved: learner_ann_model.keras/.h5, preprocessing.pkl, best_rf_model.pkl')

## 9. Inference on new students

In [ ]:
import tensorflow as tf, pickle, numpy as np
obj=pickle.load(open('preprocessing.pkl','rb')); sc=obj['scaler']; le=obj['label_encoder']
mdl=tf.keras.models.load_model('learner_ann_model.keras')
# columns: [msq, TTQ_sec, accuracy, HIT_encoded]
new=np.array([[10,200,1.0,0],[9,900,0.9,0],[5,300,0.5,3]],dtype='float32')
p=mdl.predict(sc.transform(new),verbose=0)
for r,pr in zip(new,p): print(r,'->',le.classes_[pr.argmax()],'(p=%.2f)'%pr.max())